# FWESelector

Keeps every feature whose ANOVA F-test p-value is below `alpha` divided by the
number of features (Bonferroni). The chance of keeping even one uninformative
feature is then at most `alpha`. It picks its own count.

- **Cost:** one F-test per feature.
- **Strength:** the strictest guarantee of the p-value selectors.
- **Weakness:** the cutoff tightens as features are added, so on wide data it
  keeps few features, or none.
- **Modes:** `task='classification'` or `'regression'`. Wraps scikit-learn's
  `SelectFwe`.

In [1]:
import numpy as np
from sklearn.datasets import make_classification

from featsel.selectors import FDRSelector, FWESelector

# 500 features, the first 10 informative (shuffle=False keeps them first).
X, y = make_classification(n_samples=300, n_features=500, n_informative=10,
                           n_redundant=0, shuffle=False, random_state=0)
truth = set(range(10))

## Stricter than the false discovery rate

In [2]:
for cls in (FDRSelector, FWESelector):
    chosen = set(cls(alpha=0.05).fit(X, y).get_support(indices=True))
    print(f'{cls.__name__:<12} kept {len(chosen):3d}: '
          f'{len(chosen & truth)} informative, {len(chosen - truth)} false')

FDRSelector  kept   5: 4 informative, 1 false
FWESelector  kept   3: 3 informative, 0 false


## Where it fails

The same 10 informative features, buried among more and more noise. The
Bonferroni cutoff divides `alpha` by the feature count, so only the strongest
informative features survive once the data is wide.

In [3]:
rng = np.random.default_rng(0)
for n_noise in (0, 490, 4990, 49990):
    X_wide = np.column_stack([X[:, :10], rng.normal(size=(len(y), n_noise))])
    chosen = set(FWESelector(alpha=0.05).fit(X_wide, y).get_support(indices=True))
    print(f'{X_wide.shape[1]:6d} features: {len(chosen & truth)} of 10 informative kept')

    10 features: 5 of 10 informative kept
   500 features: 3 of 10 informative kept
  5000 features: 3 of 10 informative kept


 50000 features: 3 of 10 informative kept
